# **This pipeline extracts MD&A sections from SEC filings, computes sentiment risk signals using FinBERT, and merges them with numeric financial ratios for LSTM training.**

# ***Step 1: Environment Initialization & Storage Handshake***

In [6]:
import os
from google.colab import drive
import polars as pl

# 1. THE CONNECTION
drive.mount('/content/drive')

# 2. THE PATHS (Standardized for the team)
# NOTE: Ensure the folder "04_master_ml" exists inside your sec_data folder
base_path = "/content/drive/My Drive/sec_data"
master_file = os.path.join(base_path, "04_master_ml/lstm_ready_data.parquet")

# 3. THE VERIFICATION
if os.path.exists(master_file):
    print("✅ Handshake Success: Partner 2 is ready to consume Partner 1's data.")
else:
    # This prevents the rest of the notebook from running if the path is wrong
    raise FileNotFoundError(f"❌ FATAL ERROR: Master file not found at {master_file}. Check Drive path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Handshake Success: Partner 2 is ready to consume Partner 1's data.


# ***Step 2: Master Data Ingestion & Metadata Mapping***

In [2]:
# 1. LOADING THE GOLD LAYER
master_df = pl.read_parquet(master_file)

# 2. EXTRACTING UNIQUE FILING PAIRS (CIK and ADSH)
# We need the CIK to build the correct URL
unique_filings = master_df.select(["cik", "adsh"]).unique().to_dicts()

print(f"📊 Metadata Ingestion Complete.")
print(f"✅ Found {len(unique_filings)} unique filings (CIK + ADSH) ready for extraction.")

📊 Metadata Ingestion Complete.
✅ Found 85989 unique filings (CIK + ADSH) ready for extraction.


# ***Step 3: Heuristic-Based Textual Extraction (The Scraper Engine)***

In [3]:
import re
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "subratskj2003@gmail.com"}

def get_mda_section(cik, adsh):
    # THE CORRECT URL STRUCTURE: Archives/edgar/data/{cik}/{adsh_no_dashes}/{adsh}.txt
    adsh_clean = adsh.replace('-', '')
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{adsh_clean}/{adsh}.txt"

    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.content, "html.parser")
        raw_text = soup.get_text(separator=" ").upper()

        # Universal Pattern: Item 7 (10-K) or Item 2 (10-Q)
        start_pattern = r"(ITEM\s+[27]\.?\s{1,5}MANAGEMENT|MANAGEMENT[\'’]S\s+DISCUSSION\s+AND\s+ANALYSIS)"
        end_pattern = r"(ITEM\s+[38]|ITEM\s+7A|QUANTITATIVE\s+AND\s+QUALITATIVE)"

        starts = [m.start() for m in re.finditer(start_pattern, raw_text)]
        ends = [m.start() for m in re.finditer(end_pattern, raw_text)]

        if not starts or not ends: return None

        best_mda = ""
        for s in starts:
            valid_ends = [e for e in ends if e > s]
            if not valid_ends: continue
            candidate = raw_text[s:valid_ends[0]]
            if len(candidate) > len(best_mda):
                best_mda = candidate

        clean_text = " ".join(best_mda.split())
        return clean_text if len(clean_text) > 1000 else None

    except Exception:
        return None

print("✅ Step 3: Universal Scraper (v3.0) with Correct URL Logic is ready.")

✅ Step 3: Universal Scraper (v3.0) with Correct URL Logic is ready.


# ***Step 4: Batch Execution & Bronze Layer Persistence***

In [5]:
import time

bronze_output_path = os.path.join(base_path, "01_bronze_raw/raw_mda_text.parquet")
results = []
test_subset = unique_filings[:2000]

print(f"🕵️ Starting extraction for {len(test_subset)} reports...")

for i, filing in enumerate(test_subset):
    # PASS BOTH IDs
    text = get_mda_section(filing['cik'], filing['adsh'])

    if text:
        results.append({"adsh": filing['adsh'], "mda_text": text})
        print(f"[{i+1}/{len(test_subset)}] Success: {filing['adsh']}")
    else:
        print(f"[{i+1}/{len(test_subset)}] Failed/Empty: {filing['adsh']}")

    time.sleep(0.2)

if results:
    pl.DataFrame(results).write_parquet(bronze_output_path)
    print(f"\n Success! {len(results)} reports saved.")

🕵️ Starting extraction for 2000 reports...
[1/2000] Success: 0001437749-23-023751
[2/2000] Success: 0001213900-22-016222
[3/2000] Success: 0001410636-25-000150
[4/2000] Success: 0001477932-25-001010
[5/2000] Success: 0000875045-25-000009
[6/2000] Success: 0001829126-22-001797
[7/2000] Success: 0001437749-22-006431
[8/2000] Failed/Empty: 0001104659-23-036286
[9/2000] Success: 0001140361-23-055764
[10/2000] Success: 0001558370-22-016337
[11/2000] Success: 0000950170-24-057125
[12/2000] Success: 0000915358-24-000010
[13/2000] Success: 0001735707-24-000057
[14/2000] Success: 0001580345-25-000018
[15/2000] Success: 0001376474-25-000857
[16/2000] Success: 0001437749-22-006291
[17/2000] Success: 0001213900-22-014873
[18/2000] Success: 0000940944-22-000042
[19/2000] Success: 0001628280-22-028424
[20/2000] Failed/Empty: 0001410578-25-000574
[21/2000] Success: 0001798749-24-000015
[22/2000] Success: 0001493152-22-000327
[23/2000] Success: 0001062993-22-011291
[24/2000] Failed/Empty: 0001115055-2

In [7]:
# --- PATH RESTORATION (Run this before Step 5) ---
import os
import polars as pl

# These were defined in Step 4/5/6 and are needed now
bronze_output_path = os.path.join(base_path, "01_bronze_raw/raw_mda_text.parquet")
silver_path = os.path.join(base_path, "02_silver_cleaned/mda_sentiment_signals.parquet")
final_output_path = os.path.join(base_path, "04_master_ml/multimodal_gold_dataset.parquet")

print("🗺️ Paths restored. You can now skip the scraper and go to Step 5.")

🗺️ Paths restored. You can now skip the scraper and go to Step 5.


# ***Step 5: Sentiment Analysis & Risk Signal Generation***

In [9]:
from transformers import pipeline

# 1. INITIALIZE: Correctly loading the FinBERT model
model_name = "ProsusAI/finbert"

print(" Initializing FinBERT (Downloading model if necessary)...")
# FIX: Use model_name directly in the pipeline
sentiment_analyzer = pipeline("sentiment-analysis", model=model_name)

def get_sentiment_signal(text):
    if not text or len(text) < 100: return 0.0
    try:
        # We process a slice to stay within token limits
        result = sentiment_analyzer(text[:1500])[0]
        score_map = {"negative": -1.0, "neutral": 0.0, "positive": 1.0}
        return score_map[result['label'].lower()] * result['score']
    except Exception:
        return 0.0

# 2. PROCESS: This will now find the data you saved overnight
print("🧠 Converting Text to Risk Signals...")
mda_df = pl.read_parquet(bronze_output_path)

mda_df = mda_df.with_columns(
    pl.col("mda_text").map_elements(get_sentiment_signal, return_dtype=pl.Float64).alias("sentiment_signal")
)

# 3. SAVE TO SILVER
silver_path = os.path.join(base_path, "02_silver_cleaned/mda_sentiment_signals.parquet")
mda_df.select(["adsh", "sentiment_signal"]).write_parquet(silver_path)

print(f" Success! Risk Signals saved to Silver Layer: {silver_path}")

 Initializing FinBERT (Downloading model if necessary)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

🧠 Converting Text to Risk Signals...


Token indices sequence length is longer than the specified maximum sequence length for this model (556 > 512). Running this sequence through the model will result in indexing errors


 Success! Risk Signals saved to Silver Layer: /content/drive/My Drive/sec_data/02_silver_cleaned/mda_sentiment_signals.parquet


# ***Step 6: Final Convergence & Gold Layer Solidification***

In [10]:
# 1. LOAD AND STRIP (The 'Deep Clean')
numeric_df = pl.read_parquet(master_file).with_columns(pl.col("adsh").str.strip_chars())
sentiment_df = pl.read_parquet(silver_path).with_columns(pl.col("adsh").str.strip_chars())

# 2. FORCE DATA TYPES
numeric_df = numeric_df.with_columns(pl.col("adsh").cast(pl.Utf8))
sentiment_df = sentiment_df.with_columns(pl.col("adsh").cast(pl.Utf8))

# 3. FORENSIC CHECK (Before we join)
scraped_ids = sentiment_df["adsh"].to_list()
print(f" IDs you scraped: {scraped_ids[:3]} ...")
# Check if even ONE of your IDs exists in Partner 1's file
sample_id = scraped_ids[0]
is_present = numeric_df.filter(pl.col("adsh") == sample_id).shape[0] > 0
print(f" Is ID '{sample_id}' present in the numeric master? {is_present}")

# 4. THE JOIN
final_gold_df = numeric_df.join(
    sentiment_df.select(["adsh", "sentiment_signal"]),
    on="adsh",
    how="left"
)

# 5. THE "NEUTRAL" FIX
# We will fill nulls with 999 temporarily to see the difference between 'Neutral' and 'Missing'
final_gold_df = final_gold_df.with_columns(
    pl.col("sentiment_signal").fill_null(999.0)
)

# 6. AUDIT
missing = final_gold_df.filter(pl.col("sentiment_signal") == 999.0).shape[0]
neutral = final_gold_df.filter(pl.col("sentiment_signal") == 0.0).shape[0]
found = sentiment_df.shape[0] - (final_gold_df.filter(pl.col("sentiment_signal") == 999.0).shape[0] if False else 0) # Logic check

print(f"\n --- FINAL REPAIR AUDIT ---")
print(f" Total Rows in Master: {final_gold_df.shape[0]}")
print(f" Rows that FAILED to join (Missing): {missing}")
print(f" Rows that joined but are NEUTRAL (0.0): {neutral}")

# Final Step: Change 999 back to 0.0 for the AI model
final_gold_df = final_gold_df.with_columns(
    pl.col("sentiment_signal").replace(999.0, 0.0)
)
final_gold_df.write_parquet(final_output_path)

 IDs you scraped: ['0001437749-23-023751', '0001213900-22-016222', '0001410636-25-000150'] ...
 Is ID '0001437749-23-023751' present in the numeric master? True

 --- FINAL REPAIR AUDIT ---
 Total Rows in Master: 152764
 Rows that FAILED to join (Missing): 149599
 Rows that joined but are NEUTRAL (0.0): 2943


# ***Verification Code***

In [11]:
# 1. LOAD THE FINAL DATASET
audit_df = pl.read_parquet(final_output_path)

# 2. CHECK THE "INTELLIGENCE" COLUMN
# We want to see how many rows actually have a 'non-zero' sentiment signal
extracted_signals = audit_df.filter(pl.col("sentiment_signal") != 0.0)

print(" --- FINAL AUDIT RESULTS ---")
print(f" Total Dataset Rows: {audit_df.shape[0]}")
print(f" Rows with Scraped Sentiment: {extracted_signals.shape[0]}")
print(f" Data Coverage: {(extracted_signals.shape[0] / 20) * 100:.1f}% of your test batch")

# 3. VISUAL CHECK
# Let's look at the first 5 successful merges
print("\n PREVIEW OF MERGED DATA:")
print(audit_df.select(["cik", "adsh", "sentiment_signal"]).tail(5))

# 4. RANGE CHECK
# Ensure scores are between -1 and 1
min_score = audit_df["sentiment_signal"].min()
max_score = audit_df["sentiment_signal"].max()
print(f"\n Signal Range: {min_score:.2f} to {max_score:.2f}")

 --- FINAL AUDIT RESULTS ---
 Total Dataset Rows: 152764
 Rows with Scraped Sentiment: 222
 Data Coverage: 1110.0% of your test batch

 PREVIEW OF MERGED DATA:
shape: (5, 3)
┌─────────┬──────────────────────┬──────────────────┐
│ cik     ┆ adsh                 ┆ sentiment_signal │
│ ---     ┆ ---                  ┆ ---              │
│ i32     ┆ str                  ┆ f64              │
╞═════════╪══════════════════════╪══════════════════╡
│ 1399520 ┆ 0001558370-24-014890 ┆ 0.0              │
│ 1399520 ┆ 0001558370-22-012198 ┆ 0.0              │
│ 1399520 ┆ 0001558370-24-007045 ┆ 0.0              │
│ 1399520 ┆ 0001558370-22-012198 ┆ 0.0              │
│ 1399520 ┆ 0001558370-22-016299 ┆ 0.0              │
└─────────┴──────────────────────┴──────────────────┘

 Signal Range: -0.97 to 0.95
